In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

SEED=13
D=pd.read_csv(Path('estonia_day_ahead_data.csv'))
T=pd.to_datetime(D['datetime_utc'],utc=True).dt.tz_convert('Europe/Tallinn').dt.tz_localize(None)
S=pd.DataFrame({'actual':pd.to_numeric(D['Estonia_actual'],errors='coerce').to_numpy(),'official':pd.to_numeric(D['Estonia_forecast'],errors='coerce').to_numpy()},index=T).sort_index()
S=S[~S.index.duplicated(keep='first')].asfreq('h').ffill()
X=[]
Y=[]
O=[]
Q=[]
for q in pd.date_range('2023-01-01','2025-12-31',freq='D'):
    if q not in S.index: continue
    i=S.index.get_loc(q)
    if not isinstance(i,(int,np.integer)) or i<168 or i+24>len(S): continue
    x=S['actual'].iloc[i-168:i].to_numpy(float)
    y=S['actual'].iloc[i:i+24].to_numpy(float)
    o=S['official'].iloc[i:i+24].to_numpy(float)
    if np.isfinite(x).all() and np.isfinite(y).all(): X.append(x);Y.append(y);O.append(o);Q.append(q)
X=np.asarray(X,dtype=np.float32)
Y=np.asarray(Y,dtype=np.float32)
O=np.asarray(O,dtype=np.float32)
Q=pd.DatetimeIndex(Q)
dow=Q.dayofweek.to_numpy();doy=Q.dayofyear.to_numpy();mon=Q.month.to_numpy()
E=np.c_[np.sin(2*np.pi*dow/7),np.cos(2*np.pi*dow/7),np.sin(2*np.pi*(doy-1)/365.25),np.cos(2*np.pi*(doy-1)/365.25),np.sin(2*np.pi*(mon-1)/12),np.cos(2*np.pi*(mon-1)/12),(dow>=5).astype(float)].astype(np.float32)
A=np.c_[X,E]
tr=Q.year<2025
te=Q.year==2025
p={'n_estimators':524,'learning_rate':0.0148,'max_depth':4,'min_child_weight':5.416,'subsample':0.885,'colsample_bytree':0.994,'gamma':4.374,'reg_alpha':3.10e-4,'reg_lambda':0.002879}
m=XGBRegressor(objective='reg:squarederror',random_state=SEED,n_jobs=-1,tree_method='hist',**p)
m.fit(A[tr],Y[tr])
y=Y[te]
h=m.predict(A[te])
o=O[te]
def score(a,b):
    e=b-a
    return pd.Series({'RMSE':np.sqrt(np.mean(e*e)),'MAE':np.mean(np.abs(e)),'Bias':np.mean(e),'1-MAPE':(1-np.mean(np.abs(e/a)))*100})
print(score(y,h).round(4))
print(score(y,o).round(4))